# 🧠 Brain Tumor MRI Classification — Evaluation & Grad-CAM

**Contents:**
- Confusion Matrix
- Classification Report (Precision, Recall, F1)
- ROC-AUC Curves
- Grad-CAM Heatmaps (what the model is looking at)
- Model Comparison Summary


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize

plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor'] = '#161b22'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
IMG_SIZE = 224
BATCH_SIZE = 32
TEST_DIR = '../data/Testing'
MODEL_DIR = Path('../models')
RESULTS_DIR = Path('../results')
print('✅ Setup complete')

In [ ]:
# ─── Load Models ──────────────────────────────────────────────────
cnn_model = keras.models.load_model(str(MODEL_DIR / 'custom_cnn_final.keras'))
eff_model = keras.models.load_model(str(MODEL_DIR / 'efficientnet_finetuned.keras'))

# Test generator
test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
print('Models and test data loaded ✅')

In [ ]:
# ─── Predict ──────────────────────────────────────────────────────
y_true = test_gen.classes

cnn_probs = cnn_model.predict(test_gen, verbose=1)
eff_probs = eff_model.predict(test_gen, verbose=1)

cnn_preds = np.argmax(cnn_probs, axis=1)
eff_preds = np.argmax(eff_probs, axis=1)

print(f'\nCustom CNN  Accuracy: {np.mean(cnn_preds == y_true):.4f}')
print(f'EfficientNet Accuracy: {np.mean(eff_preds == y_true):.4f}')

In [ ]:
# ─── Confusion Matrices ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (preds, title) in zip(axes, [
    (cnn_preds, 'Custom CNN'),
    (eff_preds, 'EfficientNetB0 (Fine-tuned)')
]):
    cm = confusion_matrix(y_true, preds)
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100

    sns.heatmap(
        cm_pct, annot=True, fmt='.1f', cmap='Blues',
        xticklabels=CLASSES, yticklabels=CLASSES,
        ax=ax, linewidths=0.5, cbar_kws={'label': '%'}
    )
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: results/confusion_matrices.png')

In [ ]:
# ─── Classification Report ────────────────────────────────────────
print('\n' + '='*60)
print('  CUSTOM CNN — Classification Report')
print('='*60)
print(classification_report(y_true, cnn_preds, target_names=CLASSES))

print('\n' + '='*60)
print('  EFFICIENTNETB0 — Classification Report')
print('='*60)
print(classification_report(y_true, eff_preds, target_names=CLASSES))

In [ ]:
# ─── ROC-AUC Curves ───────────────────────────────────────────────
y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3])
colors = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (probs, title) in zip(axes, [
    (cnn_probs, 'Custom CNN'),
    (eff_probs, 'EfficientNetB0')
]):
    for i, (cls, color) in enumerate(zip(CLASSES, colors)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], probs[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC={roc_auc:.3f})')

    ax.plot([0, 1], [0, 1], 'w--', lw=1, alpha=0.4)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves — {title}', fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'roc_auc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: results/roc_auc_curves.png')

In [ ]:
# ─── Grad-CAM Implementation ──────────────────────────────────────
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    """Generate Grad-CAM heatmap for an input image."""
    grad_model = keras.models.Model(
        model.inputs,
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_output, preds = grad_model(img_array)
        pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_output = last_conv_output[0]
    heatmap = last_conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index), float(tf.reduce_max(preds).numpy())

def overlay_gradcam(img_path, heatmap, alpha=0.4):
    img = cv2.imread(str(img_path))
    img = cv2.resize(img, (224, 224))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored = cm.jet(heatmap_resized)[:, :, :3]
    heatmap_colored = (heatmap_colored * 255).astype(np.uint8)
    overlaid = cv2.addWeighted(img_rgb, 1 - alpha, heatmap_colored, alpha, 0)
    return img_rgb, overlaid

print('Grad-CAM functions defined ✅')

In [ ]:
# ─── Visualize Grad-CAM for Sample Images ─────────────────────────
# Find last conv layer name in Custom CNN
last_conv = [l.name for l in cnn_model.layers if 'conv' in l.name][-1]
print(f'Last conv layer: {last_conv}')

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
fig.suptitle('Grad-CAM — What the CNN sees', fontsize=15, fontweight='bold')

for row, cls in enumerate(CLASSES):
    cls_path = Path(f'../data/Testing/{cls}')
    img_path = list(cls_path.glob('*.jpg'))[0]

    # Preprocess
    img = keras.preprocessing.image.load_img(str(img_path), target_size=(224, 224))
    img_array = keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, 0)

    # Grad-CAM
    heatmap, pred_idx, confidence = make_gradcam_heatmap(img_array, cnn_model, last_conv)
    original, overlaid = overlay_gradcam(img_path, heatmap)

    axes[row][0].imshow(original)
    axes[row][0].set_title(f'Original\n[{cls.upper()}]', fontsize=9)
    axes[row][1].imshow(heatmap, cmap='jet')
    axes[row][1].set_title('Grad-CAM Heatmap', fontsize=9)
    axes[row][2].imshow(overlaid)
    axes[row][2].set_title(f'Overlay\nPred: {CLASSES[pred_idx]} ({confidence:.1%})', fontsize=9)

    for ax in axes[row]:
        ax.axis('off')

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'gradcam_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: results/gradcam_visualization.png')

In [ ]:
# ─── Model Comparison Summary Table ────────────────────────────────
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

summary = pd.DataFrame({
    'Model': ['Custom CNN', 'EfficientNetB0 (Fine-tuned)'],
    'Test Accuracy': [
        f"{accuracy_score(y_true, cnn_preds):.4f}",
        f"{accuracy_score(y_true, eff_preds):.4f}"
    ],
    'Macro F1': [
        f"{f1_score(y_true, cnn_preds, average='macro'):.4f}",
        f"{f1_score(y_true, eff_preds, average='macro'):.4f}"
    ],
    'Params': ['~6.2M', '~4.1M'],
    'Training Strategy': ['Scratch', 'ImageNet → Fine-tuned']
})

print('\n' + '='*70)
print('  MODEL COMPARISON SUMMARY')
print('='*70)
print(summary.to_string(index=False))
summary.to_csv(str(RESULTS_DIR / 'model_comparison.csv'), index=False)
print('\n✅ Saved: results/model_comparison.csv')